# 10. NumPy for Embeddings

## What you'll learn

- **NumPy fundamentals** -- creating arrays, shapes, dtypes, and basic operations
- **Vector operations** -- dot products, norms, and what they mean geometrically
- **Cosine similarity** -- implement it from scratch and understand why it matters for text similarity
- **Vectorized similarity** -- compute similarity across an entire matrix of embeddings efficiently
- **Top-K retrieval** -- find the most relevant documents given a query embedding
- **Mini-RAG pipeline** -- build a `retrieve()` function that powers retrieval-augmented generation

## Prerequisites

This is the final appendix notebook. You should be comfortable with:

- Python fundamentals ([Appendix 01](01_python_fundamentals.ipynb))
- Functions and scope ([Appendix 02](02_functions_and_scope.ipynb))
- Data structures -- especially lists and dicts ([Appendix 03](03_data_structures.ipynb))
- Strings and JSON ([Appendix 04](04_strings_and_json.ipynb))
- Classes and OOP ([Appendix 07](07_classes_and_oop.ipynb))
- Type hints ([Appendix 08](08_decorators_and_type_hints.ipynb))

## Why this matters for agents

When an agent needs to answer questions about documents, code, or any large body of text, it
cannot fit everything into its context window. The solution is **retrieval-augmented generation
(RAG)**: convert text chunks into numerical vectors (embeddings), then find the most relevant
chunks by computing vector similarity. NumPy is the foundation for all of this -- every
embedding library and vector database ultimately does the same linear algebra operations
you'll learn here.

In [Core 10 (rag_from_scratch)](../core/10_rag_from_scratch.ipynb) you'll use a real
embedding API. This notebook gives you the math intuition so that when you get there,
the retrieval step isn't a black box.

> **Further reading:**
> - [NumPy Quickstart Tutorial](https://numpy.org/doc/stable/user/quickstart.html) -- the official starting point for NumPy
> - [3Blue1Brown -- Essence of Linear Algebra](https://www.youtube.com/playlist?list=PLZHQObOWTQDPD3MizzM2xVFitgF8hE_ab) -- gorgeous visual intuition for vectors, dot products, and transformations
> - [Lilian Weng -- LLM Powered Autonomous Agents](https://lilianweng.github.io/posts/2023-06-23-agent/) -- see the RAG section for how retrieval fits into agent architectures

---

## 1. NumPy Basics

NumPy's core object is the **ndarray** -- an n-dimensional array of numbers, all the same
type. Unlike Python lists, NumPy arrays support fast element-wise operations and linear
algebra out of the box.

Let's start with the basics: creating arrays and inspecting their properties.

In [ ]:
import numpy as np

# Create arrays from Python lists
message_lengths = np.array([42, 105, 78, 230, 15])
print("Message lengths:", message_lengths)
print("Shape:", message_lengths.shape)   # (5,) -- 1D array with 5 elements
print("Dtype:", message_lengths.dtype)   # int64 -- NumPy chose the type
print("Mean:", message_lengths.mean())   # built-in aggregation

### Common array constructors

In agent code you'll often need to initialize embedding matrices, zero-vectors for
accumulators, or random vectors for testing. Here are the essential constructors.

In [ ]:
# Zeros and ones -- handy for initialization
zero_embedding = np.zeros(8)          # 8-dimensional zero vector
ones_vector = np.ones(8)              # 8-dimensional ones vector

# Random vectors -- we'll use these to simulate embeddings
np.random.seed(42)  # reproducibility!
random_embedding = np.random.randn(8)  # 8D vector from standard normal

print("Zeros:", zero_embedding)
print("Ones: ", ones_vector)
print("Random:", random_embedding)
print()
print("Shape of random_embedding:", random_embedding.shape)
print("Dtype of random_embedding:", random_embedding.dtype)

### 2D arrays (matrices)

When you have multiple embeddings (one per text chunk), you store them in a **matrix**
where each row is one embedding vector. This is how every vector database stores its data.

In [ ]:
np.random.seed(42)

# 5 text chunks, each with an 8-dimensional embedding
embedding_matrix = np.random.randn(5, 8)

print("Shape:", embedding_matrix.shape)  # (5, 8) -- 5 rows, 8 columns
print("Number of chunks:", embedding_matrix.shape[0])
print("Embedding dimension:", embedding_matrix.shape[1])
print()
print("First embedding (row 0):")
print(embedding_matrix[0])
print()
print("All embeddings, first 3 dims (column slice):")
print(embedding_matrix[:, :3])

---

## 2. Vector Operations: Dot Product and Norm

Two operations are at the heart of embedding similarity:

- **Dot product** (`a . b`): measures how much two vectors point in the same direction.
  Large positive = similar direction. Near zero = unrelated. Negative = opposite.
- **Norm** (`||a||`): the length/magnitude of a vector. We need this to normalize
  vectors so we can compare direction independent of magnitude.

Think of it this way: if two text chunks are about the same topic, their embedding
vectors will point in roughly the same direction in the high-dimensional space.

In [ ]:
np.random.seed(42)

# Two vectors about "similar" topics (we fake this with close values)
base = np.random.randn(8)
similar = base + np.random.randn(8) * 0.1   # small perturbation
different = np.random.randn(8)               # completely unrelated

# Dot product
print("Dot product (base vs similar): ", np.dot(base, similar))
print("Dot product (base vs different):", np.dot(base, different))
print()

# Norm (Euclidean length)
print("Norm of base:     ", np.linalg.norm(base))
print("Norm of similar:  ", np.linalg.norm(similar))
print("Norm of different:", np.linalg.norm(different))

Notice how the dot product is much higher for `base` vs `similar` than for `base` vs
`different`. But there's a problem: the dot product also depends on the *magnitude* of
the vectors. Two long vectors about different topics could have a higher dot product
than two short vectors about the same topic. We need to **normalize**.

---

## 3. Cosine Similarity (From Scratch)

**Cosine similarity** fixes the magnitude problem by dividing the dot product by the
product of the norms:

$$\text{cosine\_sim}(a, b) = \frac{a \cdot b}{\|a\| \cdot \|b\|}$$

The result is always between -1 (opposite) and 1 (identical direction), regardless of
vector length. This is the standard similarity metric for text embeddings.

Let's implement it from scratch.

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Compute cosine similarity between two vectors."""
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return float(dot_product / (norm_a * norm_b))

In [ ]:
np.random.seed(42)

base = np.random.randn(8)
similar = base + np.random.randn(8) * 0.1
different = np.random.randn(8)
opposite = -base  # exactly opposite direction

print("base vs similar:  ", f"{cosine_similarity(base, similar):.4f}")   # ~0.99
print("base vs different:", f"{cosine_similarity(base, different):.4f}") # near 0
print("base vs opposite: ", f"{cosine_similarity(base, opposite):.4f}")  # -1.0
print("base vs itself:   ", f"{cosine_similarity(base, base):.4f}")      # 1.0

This is exactly what happens inside every vector database when you search for similar
documents. The embedding model converts text to vectors, and the database computes
cosine similarity between your query vector and every stored document vector.

---

## 4. Matrix of Embeddings: Vectorized Similarity

In practice, you don't compare one vector at a time. You have a **matrix** of document
embeddings and want to compute similarity between a query and *all* documents at once.
NumPy's vectorized operations make this fast.

Let's create a realistic scenario: an agent has indexed 10 text chunks about different
topics. We simulate embeddings where related chunks have similar vectors.

In [ ]:
np.random.seed(99)

# Simulate 10 text chunks about 3 topics:
# Topic A (agents): chunks 0, 1, 2
# Topic B (tools):  chunks 3, 4, 5
# Topic C (memory): chunks 6, 7, 8, 9

# Base vectors for each topic (these define the "direction" of each topic)
topic_a_base = np.array([1.0, 0.5, -0.3, 0.8, 0.2, -0.1, 0.4, 0.6])
topic_b_base = np.array([-0.5, 1.0, 0.7, -0.2, 0.9, 0.3, -0.4, 0.1])
topic_c_base = np.array([0.2, -0.6, 1.0, 0.1, -0.3, 0.8, 0.5, -0.7])

# Each chunk = topic base + small random noise (simulates slight differences)
noise_scale = 0.15
embeddings = np.vstack([
    topic_a_base + np.random.randn(8) * noise_scale,  # chunk 0
    topic_a_base + np.random.randn(8) * noise_scale,  # chunk 1
    topic_a_base + np.random.randn(8) * noise_scale,  # chunk 2
    topic_b_base + np.random.randn(8) * noise_scale,  # chunk 3
    topic_b_base + np.random.randn(8) * noise_scale,  # chunk 4
    topic_b_base + np.random.randn(8) * noise_scale,  # chunk 5
    topic_c_base + np.random.randn(8) * noise_scale,  # chunk 6
    topic_c_base + np.random.randn(8) * noise_scale,  # chunk 7
    topic_c_base + np.random.randn(8) * noise_scale,  # chunk 8
    topic_c_base + np.random.randn(8) * noise_scale,  # chunk 9
])

print("Embedding matrix shape:", embeddings.shape)  # (10, 8)

**Note on synthetic embeddings:** Real embedding models (like OpenAI's `text-embedding-3-small` or open-source `sentence-transformers`) learn these topic clusters from training data — text about similar concepts naturally ends up with similar vectors. Here we're simulating that result with base vectors + noise so we can focus on the math without needing an API call.

def cosine_similarity_matrix(query: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    """Compute cosine similarity between a query vector and every row in a matrix.

    Args:
        query: 1D array of shape (d,)
        matrix: 2D array of shape (n, d)

    Returns:
        1D array of shape (n,) with similarity scores
    """
    # Normalize the query to unit length
    query_norm = query / np.linalg.norm(query)
    # Normalize each row of the matrix to unit length
    row_norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    matrix_norm = matrix / row_norms
    # @ is matrix multiplication. Since both sides are unit-normalized,
    # each dot product equals the cosine similarity between that row and
    # the query. This computes all n similarities in one operation.
    return matrix_norm @ query_norm

In [ ]:
def cosine_similarity_matrix(query: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    """Compute cosine similarity between a query vector and every row in a matrix.

    Args:
        query: 1D array of shape (d,)
        matrix: 2D array of shape (n, d)

    Returns:
        1D array of shape (n,) with similarity scores
    """
    # Normalize the query
    query_norm = query / np.linalg.norm(query)
    # Normalize each row of the matrix
    row_norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    matrix_norm = matrix / row_norms
    # Dot product of normalized query with each normalized row
    return matrix_norm @ query_norm

In [ ]:
# Query: something about agents (should be most similar to chunks 0, 1, 2)
query = topic_a_base + np.random.randn(8) * noise_scale

similarities = cosine_similarity_matrix(query, embeddings)

print("Similarities to each chunk:")
for i, sim in enumerate(similarities):
    topic = "agents" if i < 3 else "tools" if i < 6 else "memory"
    marker = " <--" if sim > 0.9 else ""
    print(f"  Chunk {i:2d} ({topic:>6s}): {sim:.4f}{marker}")

The agent-topic chunks (0, 1, 2) should have the highest similarity scores. This is
exactly how a RAG pipeline decides which chunks to inject into the LLM's context.

---

## 5. Top-K Retrieval

In a real RAG system, you don't return *all* chunks -- you return the **top K** most
similar ones. NumPy's `argsort` gives us the indices that would sort an array, so we
can grab the top-K indices efficiently.

In [ ]:
def top_k_indices(similarities: np.ndarray, k: int = 3) -> np.ndarray:
    """Return indices of the top-k highest similarity scores."""
    # argsort returns indices in ascending order; we want descending
    sorted_indices = np.argsort(similarities)[::-1]
    return sorted_indices[:k]

In [ ]:
# Retrieve top 3 chunks for our agents-related query
top_3 = top_k_indices(similarities, k=3)

print("Top 3 most similar chunks:")
for rank, idx in enumerate(top_3, 1):
    print(f"  #{rank}: Chunk {idx} (similarity: {similarities[idx]:.4f})")

print()

# Now try a tools-related query
tools_query = topic_b_base + np.random.randn(8) * noise_scale
tools_sims = cosine_similarity_matrix(tools_query, embeddings)
top_3_tools = top_k_indices(tools_sims, k=3)

print("Top 3 for tools-related query:")
for rank, idx in enumerate(top_3_tools, 1):
    print(f"  #{rank}: Chunk {idx} (similarity: {tools_sims[idx]:.4f})")

---

## 6. Connecting to Text Chunks

So far we've worked with raw numbers. In a real agent, each embedding row corresponds
to an actual text chunk. Let's connect the math to real text.

Below we simulate what an embedding API would produce: each text chunk gets a vector,
and chunks about the same topic get similar vectors.

In [ ]:
# Text chunks an agent might have indexed from documentation
chunks = [
    "An agent is a system that uses an LLM to decide actions in a loop.",
    "The core agent loop: observe, think, act, repeat until done.",
    "ReAct agents interleave reasoning and acting steps.",
    "Tools let agents interact with external systems like APIs and databases.",
    "A tool definition includes a name, description, and parameter schema.",
    "Function calling is how the LLM specifies which tool to use and with what arguments.",
    "Short-term memory holds the current conversation history.",
    "Long-term memory uses a vector store to persist knowledge across sessions.",
    "RAG retrieves relevant documents to augment the LLM's context window.",
    "Embedding models convert text into dense vectors that capture semantic meaning.",
]

print(f"Number of chunks: {len(chunks)}")
print(f"Number of embeddings: {embeddings.shape[0]}")
assert len(chunks) == embeddings.shape[0], "Mismatch!"

In [ ]:
def retrieve(query_emb: np.ndarray,
             chunks: list[str],
             chunk_embs: np.ndarray,
             k: int = 3) -> list[dict]:
    """Retrieve the top-k most similar chunks for a query embedding.

    Returns a list of dicts with 'chunk', 'index', and 'score' keys.
    """
    similarities = cosine_similarity_matrix(query_emb, chunk_embs)
    top_indices = top_k_indices(similarities, k=k)
    return [
        {"chunk": chunks[i], "index": int(i), "score": float(similarities[i])}
        for i in top_indices
    ]

In [ ]:
# Simulate a query about agents
np.random.seed(123)
agent_query_emb = topic_a_base + np.random.randn(8) * noise_scale

results = retrieve(agent_query_emb, chunks, embeddings, k=3)

print("Query: 'How do agents work?'")
print("Retrieved chunks:\n")
for r in results:
    print(f"  [{r['score']:.4f}] {r['chunk']}")

In [ ]:
# Simulate a query about memory
np.random.seed(456)
memory_query_emb = topic_c_base + np.random.randn(8) * noise_scale

results = retrieve(memory_query_emb, chunks, embeddings, k=3)

print("Query: 'How does agent memory work?'")
print("Retrieved chunks:\n")
for r in results:
    print(f"  [{r['score']:.4f}] {r['chunk']}")

This is the complete retrieval step of RAG. In a real system, you'd replace the
synthetic embeddings with vectors from an embedding API (like OpenAI's
`text-embedding-3-small` or a local model), but the math is identical.

---

## Putting It Together: Mini-RAG Retrieval System

Let's combine everything into a clean, self-contained mini-RAG pipeline. This is the
kind of code you'll refactor into a proper tool when building RAG agents in the core
track.

The flow:
1. Start with text chunks (from a document, web page, codebase, etc.)
2. Each chunk has an embedding (in production, from an embedding API)
3. A user query comes in -- compute its embedding
4. Find the top-K most similar chunks
5. Return them as context for the LLM

In [ ]:
def make_synthetic_embeddings(n_chunks: int,
                              dim: int,
                              topic_bases: list[np.ndarray],
                              topic_assignments: list[int],
                              noise: float = 0.15,
                              seed: int = 42) -> np.ndarray:
    """Generate synthetic embeddings where chunks in the same topic cluster together.

    Args:
        n_chunks: number of chunks
        dim: embedding dimension
        topic_bases: list of base vectors, one per topic
        topic_assignments: which topic each chunk belongs to (list of indices)
        noise: scale of random perturbation
        seed: random seed for reproducibility
    """
    np.random.seed(seed)
    embs = np.zeros((n_chunks, dim))
    for i, topic_idx in enumerate(topic_assignments):
        embs[i] = topic_bases[topic_idx] + np.random.randn(dim) * noise
    return embs

In [ ]:
# --- The mini-RAG system ---

# Step 1: Our knowledge base
knowledge_chunks = [
    "An agent is a system that uses an LLM to decide actions in a loop.",
    "The core agent loop: observe, think, act, repeat until done.",
    "ReAct agents interleave reasoning and acting steps.",
    "Tools let agents interact with external systems like APIs and databases.",
    "A tool definition includes a name, description, and parameter schema.",
    "Function calling is how the LLM specifies which tool to use.",
    "Short-term memory holds the current conversation history.",
    "Long-term memory uses a vector store to persist knowledge.",
    "RAG retrieves relevant documents to augment the LLM's context.",
    "Embedding models convert text into dense vectors for similarity search.",
]

# Step 2: Topic structure (3 topics, assigned to chunks)
topic_bases = [
    np.array([1.0, 0.5, -0.3, 0.8, 0.2, -0.1, 0.4, 0.6]),   # agents
    np.array([-0.5, 1.0, 0.7, -0.2, 0.9, 0.3, -0.4, 0.1]),   # tools
    np.array([0.2, -0.6, 1.0, 0.1, -0.3, 0.8, 0.5, -0.7]),   # memory/RAG
]
# chunks 0-2 = agents, 3-5 = tools, 6-9 = memory/RAG
assignments = [0, 0, 0, 1, 1, 1, 2, 2, 2, 2]

# Step 3: Generate embeddings
chunk_embeddings = make_synthetic_embeddings(
    n_chunks=len(knowledge_chunks),
    dim=8,
    topic_bases=topic_bases,
    topic_assignments=assignments,
    seed=77,
)

print(f"Knowledge base: {len(knowledge_chunks)} chunks")
print(f"Embedding matrix: {chunk_embeddings.shape}")

In [ ]:
# Step 4: Simulate queries and retrieve

queries = [
    ("What is an agent?", 0),           # agents topic
    ("How do tools work?", 1),           # tools topic
    ("Tell me about RAG and memory", 2), # memory topic
]

np.random.seed(999)
for query_text, topic_idx in queries:
    # In production this would be: query_emb = embedding_api.embed(query_text)
    query_emb = topic_bases[topic_idx] + np.random.randn(8) * 0.15

    results = retrieve(query_emb, knowledge_chunks, chunk_embeddings, k=3)

    print(f"Query: '{query_text}'")
    for r in results:
        print(f"  [{r['score']:.4f}] {r['chunk']}")
    print()

Each query correctly retrieves chunks from its own topic cluster. In a real
RAG agent, these retrieved chunks would be injected into the system prompt:

```python
# Pseudocode for how RAG feeds into an agent
context_chunks = retrieve(query_emb, chunks, embeddings, k=3)
context_text = "\n".join(r["chunk"] for r in context_chunks)

messages = [
    {"role": "system", "content": f"Answer using this context:\n{context_text}"},
    {"role": "user", "content": user_question},
]
response = call_llm(messages)
```

That's the complete RAG pipeline -- and the math underneath it is exactly what
you built in this notebook.

# Exercise 1: Your code here

# a = np.array([1.0, 2.0, 3.0])
# b = np.array([4.0, 5.0, 6.0])
#
# Step 1: dot product  -> 1*4 + 2*5 + 3*6 = 4 + 10 + 18 = 32
# Step 2: norm(a)      -> sqrt(1 + 4 + 9)    = sqrt(14) ≈ 3.7417
#         norm(b)      -> sqrt(16 + 25 + 36)  = sqrt(77) ≈ 8.7749
# Step 3: sim          -> 32 / (3.7417 * 8.7749) ≈ 0.974632
#
# dot = ???
# norm_a = ???
# norm_b = ???
# sim = ???
# print(f"Cosine similarity: {sim:.6f}")  # Expected: 0.974632

In [ ]:
# Exercise 1: Your code here


# Exercise 2: Your code here

np.random.seed(42)
matrix = np.random.randn(50, 8)

# Step 1: Normalize rows to unit length
norms = np.linalg.norm(matrix, axis=1, keepdims=True)
matrix_normed = matrix / norms

# Step 2: Compute full 50x50 similarity matrix
# (matrix multiplication of normed rows: each dot product = cosine similarity)
sim_matrix = matrix_normed @ matrix_normed.T

# Step 3: Mask the diagonal (self-similarity is always 1.0, not interesting)
np.fill_diagonal(sim_matrix, np.nan)

# Step 4: Find the most and least similar pairs
# TODO: use np.nanargmax(sim_matrix) to get the flat index of the max value,
#       then np.unravel_index(flat_index, sim_matrix.shape) to convert it
#       back to (row, col) coordinates. Same for np.nanargmin.
#
# most_similar_flat = np.nanargmax(sim_matrix)
# most_i, most_j = np.unravel_index(most_similar_flat, sim_matrix.shape)
# print(f"Most similar: rows {most_i} and {most_j} (sim={sim_matrix[most_i, most_j]:.4f})")
#
# least_similar_flat = np.nanargmin(sim_matrix)
# least_i, least_j = np.unravel_index(least_similar_flat, sim_matrix.shape)
# print(f"Least similar: rows {least_i} and {least_j} (sim={sim_matrix[least_i, least_j]:.4f})")

In [ ]:
# Exercise 2: Your code here


### Exercise 3: VectorStore Class

Build a `VectorStore` class that wraps everything from this notebook into a clean
interface. It should support:

- `add(text: str, embedding: np.ndarray)` -- add a chunk and its embedding
- `search(query_embedding: np.ndarray, k: int = 3) -> list[dict]` -- return top-k results
- `__len__()` -- return the number of stored chunks

```python
class VectorStore:
    def __init__(self):
        self.texts = []
        self.embeddings = None  # will be a numpy matrix

    def add(self, text: str, embedding: np.ndarray) -> None:
        ???

    def search(self, query_embedding: np.ndarray, k: int = 3) -> list[dict]:
        ???

    def __len__(self) -> int:
        ???

# Test it:
store = VectorStore()
np.random.seed(42)
for chunk in knowledge_chunks:
    emb = np.random.randn(8)  # fake embedding
    store.add(chunk, emb)

print(f"Store has {len(store)} chunks")
results = store.search(np.random.randn(8), k=2)
for r in results:
    print(f"  [{r['score']:.4f}] {r['text']}")
```

In [ ]:
# Exercise 3: Your code here


### Exercise 4 (Stretch): Threshold-Based Retrieval

Modify the `retrieve` function (or your `VectorStore.search` method) to accept an
optional `threshold` parameter. When set, only return chunks with similarity **above**
the threshold, even if that means returning fewer than `k` results. This is useful in
agents -- if nothing in the knowledge base is relevant, it's better to return nothing
than to return garbage context.

In [ ]:
# Exercise 4: Your code here


---

## Summary

In this notebook you learned:

| Concept | NumPy operation | Why it matters for agents |
|---------|----------------|---------------------------|
| Array creation | `np.array`, `np.zeros`, `np.random.randn` | Initialize and store embeddings |
| Dot product | `np.dot(a, b)` | Core of similarity computation |
| Vector norm | `np.linalg.norm(a)` | Normalize vectors for fair comparison |
| Cosine similarity | `dot / (norm_a * norm_b)` | Standard metric for text similarity |
| Matrix operations | `matrix @ vector` | Vectorized similarity over all documents |
| Top-K retrieval | `np.argsort(...)[::-1][:k]` | Find the most relevant chunks |

You're now ready for [Core 10: RAG from Scratch](../core/10_rag_from_scratch.ipynb),
where you'll swap these synthetic embeddings for real ones from an embedding API
and build a complete retrieval-augmented generation pipeline for an agent.

> **Further reading:**
> - [NumPy Quickstart Tutorial](https://numpy.org/doc/stable/user/quickstart.html) -- official guide to array fundamentals
> - [3Blue1Brown -- Essence of Linear Algebra](https://www.youtube.com/playlist?list=PLZHQObOWTQDPD3MizzM2xVFitgF8hE_ab) -- if you want deep geometric intuition
> - [Lilian Weng -- LLM Powered Autonomous Agents](https://lilianweng.github.io/posts/2023-06-23-agent/) -- the RAG section explains how retrieval fits into the agent architecture